In [ ]:
from sklearn.cluster import KMeans
import numpy as np
import num2words
import string
import contractions
from collections import Counter
from spacy.lang.en.stop_words import STOP_WORDS

In [7]:
# Class 1: Sports & Athletics (Context: Winning/Medals)
doc1 = "The gold medal price is high effort"
doc2 = "Winning a gold medal needs a high jump"
doc3 = "Market for a gold medal is a trade of sweat"
doc4 = "The athlete will trade all for a gold medal"

# Class 2: Finance & Economy (Context: Market/Investment)
doc5 = "The gold bars price is high today"
doc6 = "Investing in gold bars needs a high rate"
doc7 = "Market for gold bars is a trade of money"
doc8 = "The bank will trade all for gold bars"


def preprocess_text(text):
    # kowercase
    text = text.lower()

    #expand contractions
    text = contractions.fix(text)

    # remove punctuation
    text = ''.join(ch for ch in text if ch not in string.punctuation)

    #tokenize
    tokens = text.split()

    # normalize numbers + drop stop words
    result = []
    for tok in tokens:
        if tok in STOP_WORDS:
            continue
        if tok.isdigit():
            tok = num2words.num2words(int(tok))
        result.append(tok)

    return result

def vectorize(docs, n_gram_size=1):
    def get_ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

    preprocessed = [preprocess_text(doc) for doc in docs]
    
    all_ngrams = []
    for doc in preprocessed:
        all_ngrams += get_ngrams(doc, n_gram_size)
    
    vocab = set(all_ngrams)
    vocab = sorted(vocab)  # sorting this for better result reproduciing

    vectors = []
    for doc in preprocessed:
        ngrams = get_ngrams(doc, n_gram_size)
        counts = Counter(ngrams)
        vector = [counts.get(ngram, 0) for ngram in vocab]
        vectors.append(vector)

    return np.array(vectors)

# Training / Clustering

all_docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8]

# quick sanity check of preprocessing
for d in all_docs:
    print(d, "->", preprocess_text(d))

# 1-gram Experiment
X1 = vectorize(all_docs, n_gram_size=1)
km1 = KMeans(n_clusters=2, random_state=42).fit(X1)

# 2-gram Experiment
X2 = vectorize(all_docs, n_gram_size=2)
km2 = KMeans(n_clusters=2, random_state=42).fit(X2)

print(f"1-gram clusters: {km1.labels_}")
print(f"2-gram clusters: {km2.labels_}")

# compare accuracy and precision

The gold medal price is high effort -> ['gold', 'medal', 'price', 'high', 'effort']
Winning a gold medal needs a high jump -> ['winning', 'gold', 'medal', 'needs', 'high', 'jump']
Market for a gold medal is a trade of sweat -> ['market', 'gold', 'medal', 'trade', 'sweat']
The athlete will trade all for a gold medal -> ['athlete', 'trade', 'gold', 'medal']
The gold bars price is high today -> ['gold', 'bars', 'price', 'high', 'today']
Investing in gold bars needs a high rate -> ['investing', 'gold', 'bars', 'needs', 'high', 'rate']
Market for gold bars is a trade of money -> ['market', 'gold', 'bars', 'trade', 'money']
The bank will trade all for gold bars -> ['bank', 'trade', 'gold', 'bars']
1-gram clusters: [0 1 0 0 1 1 0 0]
2-gram clusters: [0 0 0 1 1 1 1 1]


In [ ]:
# Training / Clustering

all_docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8]

# 1-gram Experiment
X1 = vectorize(all_docs, n_gram_size=1)
km1 = KMeans(n_clusters=2, random_state=42).fit(X1)

# 2-gram Experiment
X2 = vectorize(all_docs, n_gram_size=2)
km2 = KMeans(n_clusters=2, random_state=42).fit(X2)

print(f"1-gram clusters: {km1.labels_}")
print(f"2-gram clusters: {km2.labels_}")

# compare accuracy and precision

The gold medal price is high effort -> ['gold', 'medal', 'price', 'high', 'effort']
Winning a gold medal needs a high jump -> ['winning', 'gold', 'medal', 'needs', 'high', 'jump']
Market for a gold medal is a trade of sweat -> ['market', 'gold', 'medal', 'trade', 'sweat']
The athlete will trade all for a gold medal -> ['athlete', 'trade', 'gold', 'medal']
The gold bars price is high today -> ['gold', 'bars', 'price', 'high', 'today']
Investing in gold bars needs a high rate -> ['investing', 'gold', 'bars', 'needs', 'high', 'rate']
Market for gold bars is a trade of money -> ['market', 'gold', 'bars', 'trade', 'money']
The bank will trade all for gold bars -> ['bank', 'trade', 'gold', 'bars']
1-gram clusters: [0 1 0 0 1 1 0 0]
2-gram clusters: [0 0 0 1 1 1 1 1]


## Task 2

In [16]:
# Documents
D1 = "I love cats"
D2 = "Cats are chill"
D3 = "I am late"

# Your Task: implement context window vectorization
# with window size = 1 (so each window is 3 tokens wide)
# Use <s> and </s> padding flags

def add_padding(tokens):
    return ["<s>"] + tokens + ["</s>"]

def extract_windows(tokens, window_size=1):
    windows = []
    for i in range(len(tokens) - 2*window_size + 1 + 1):
        window = tuple(tokens[i:i+2*window_size+1])
        windows.append(window)
    return windows

def build_vocab(all_windows):
    vocab = set(all_windows)
    vocab = sorted(vocab) 
    vocab_index = {w: i for i, w in enumerate(vocab)}
    return vocab, vocab_index

def vectorize_doc(doc_windows, vocab):
    doc_set = set(doc_windows)
    return [1 if w in doc_set else 0 for w in vocab]
# Run
all_docs = [D1, D2, D3]
all_windows = []
docs_windows = []


for doc in all_docs:
    tokens = preprocess_text(doc)
    tokens = add_padding(tokens)
    windows = extract_windows(tokens, window_size=1)

    docs_windows.append(windows)
    all_windows.extend(windows)

vocab, vocab_index = build_vocab(all_windows)

vectors = [vectorize_doc(w, vocab) for w in docs_windows]

print("VOCAB:")
for i, v in enumerate(vocab):
    print(i, v)

print("\nVECTORS:")
for v in vectors:
    print(v)

VOCAB:
0 ('</s>',)
1 ('<s>', 'cats', 'chill')
2 ('<s>', 'late', '</s>')
3 ('<s>', 'love', 'cats')
4 ('cats', '</s>')
5 ('cats', 'chill', '</s>')
6 ('chill', '</s>')
7 ('late', '</s>')
8 ('love', 'cats', '</s>')

VECTORS:
[1, 0, 0, 1, 1, 0, 0, 0, 1]
[1, 1, 0, 0, 0, 1, 1, 0, 0]
[1, 0, 1, 0, 0, 0, 0, 1, 0]


## Hello
sorry for late submition


for clarity and honesty, every line that has tuple/sorted is from a stackoverflow+ai debugging session